<a href="https://colab.research.google.com/github/SebastianRodriguez05/Teoria_de_Aprendizaje_de_Maquinas-/blob/main/Proyecto_final/Proyecto_tam.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
  #instalación de librerías
!pip install streamlit -q
!pip install --force-reinstall https://github.com/yt-dlp/yt-dlp/archive/master.tar.gz #Pegar los links para el github

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 62.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 59.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.1/79.1 kB 5.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 5.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for yt-dlp: filename=yt_dlp-2025.7.21-py3-none-any.whl size=3025415 sha256=05a2e7730f038216ad8f6a22b632df6eb011469adb09f6afb325fb1848b9a805
  Stored in directory: /tmp/pip-ephem-wheel-cache-ey9axl8z/wheels/2d/79/97/7209650ef73114e0fe0603480da012ad3afacb9cae6b8acd9a
Successfully built yt-dlp


In [2]:
!mkdir pages

# **Página principal**

In [9]:
%%writefile 0_👋_BIENVENIDOS.py
import streamlit as st

# Configuración de la página
st.set_page_config(
    page_title="Proyecto final- Teoría de Aprendizaje de Máquinas",
    page_icon="📚",
    layout="centered"
)

# Título principal
st.title("📚 Proyecto final - Teoría de Aprendizaje de Máquinas")

# Subtítulo y descripción del proyecto
st.markdown("""
¡Bienvenid@ al tablero interactivo del **Proyecto final** de la asignatura **Teoría de Aprendizaje de Máquinas**! 🎓

Este espacio ha sido diseñado para **explorar y visualizar** el rendimiento de distintos modelos de clasificación y técnicas de reducción de dimensionalidad aplicadas al **dataset de dígitos manuscritos (USPS)**.

---

🔎 **¿Qué encontrarás aquí?**
- Comparación de clasificadores como **Regresión Logística**, **Random Forest** y **MLP**.
- Análisis de representaciones 2D usando **PCA** y **UMAP**.
- Curvas **ROC**, **matrices de confusión** y reportes de clasificación.
- Visualizaciones interactivas y estilizadas para facilitar la interpretación de resultados.

Utiliza el **menú lateral** para navegar entre las diferentes secciones del proyecto.

""")

# Barra lateral
st.sidebar.success("⬅️ Selecciona una sección para comenzar tu análisis.")

# Créditos o pie de página
st.markdown("""
---
🧠 **Créditos**
Este tablero fue desarrollado como parte del **Parcial 2** del curso **Teoría de Aprendizaje de Máquinas**.

💻 Desarrollado con: [Streamlit](https://streamlit.io)
""")


Overwriting 0_👋_BIENVENIDOS.py


# **Páginas**

In [4]:
%%writefile 1_Regression.py

import streamlit as st
import pandas as pd
import gdown
import os
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

# Título principal
st.title("🎵 Predicción de Popularidad Musical con Modelos de Regresión")

# -------------------------------------------
st.header("1. Carga de datos desde Google Drive")

output_path = 'cleaned_music_population.csv'
file_id = '1jpW7VNRftEUjVj_yZCOBY6SgA5qa6J4T'
url = f'https://drive.google.com/uc?id={file_id}'

if not os.path.exists(output_path):
    st.info("📥 Descargando archivo desde Google Drive...")
    gdown.download(url, output_path, quiet=False)

df = pd.read_csv(output_path)
st.success("✅ Archivo cargado correctamente.")
st.write("Vista previa del DataFrame original:")
st.dataframe(df.head())

# -------------------------------------------
st.header("2. Preprocesamiento")

df_onehot = pd.get_dummies(df, drop_first=True)
st.write("🔍 Shape después del one-hot encoding:", df_onehot.shape)
st.write("📏 Shape original:", df.shape)

df_onehot = df_onehot[df_onehot['popularity'] > 0]

X = df_onehot.drop(columns=['popularity'])
y = df_onehot['popularity']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# -------------------------------------------
st.header("3. Modelos de Regresión")

# =============================
st.subheader("🔵 Modelo: Regresión Lineal")
model = LinearRegression()
model.fit(X_train_scaled, y_train)
y_pred = model.predict(X_test_scaled)

df_results = X_test.copy()
df_results['popularity_real'] = y_test
df_results['popularity_pred'] = y_pred.round(2)

st.write("🔍 Primeras 10 predicciones:")
st.dataframe(df_results.head(10))

with st.expander("📉 Coeficientes del modelo"):
    st.write(f"Intercepto: {model.intercept_:.2f}")
    for idx, col_name in enumerate(X.columns):
        st.write(f"{col_name}: {model.coef_[idx]:.4f}")

# Gráfico de dispersión
df_results['music_genre'] = df.loc[df_results.index, 'music_genre']
fig, ax = plt.subplots(figsize=(10, 6))
sns.scatterplot(data=df_results, x='popularity_real', y='popularity_pred', hue='music_genre', ax=ax)
ax.set_title("Real vs Predicho - Regresión Lineal")
ax.set_xlabel("Popularidad Real")
ax.set_ylabel("Popularidad Predicha")
st.pyplot(fig)

# Métricas
mse = mean_squared_error(y_test, y_pred)
rmse = mse ** 0.5
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)

st.markdown("**📊 Métricas:**")
st.write(f"📉 MSE: {mse:.2f}")
st.write(f"📉 RMSE: {rmse:.2f}")
st.write(f"📈 R²: {r2:.2f}")
st.write(f"📉 MAE: {mae:.2f}")

# Histograma de residuos
fig, ax = plt.subplots(figsize=(10, 6))
sns.histplot(y_test - y_pred, bins=15, kde=True, ax=ax)
ax.set_title("Distribución de los residuos - Regresión Lineal")
st.pyplot(fig)

# =============================
st.subheader("🌳 Modelo: Árbol de Decisión")
dt_model = DecisionTreeRegressor()
dt_model.fit(X_train_scaled, y_train)
y_pred_dt = dt_model.predict(X_test_scaled)

st.write("🔍 Primeras 10 predicciones:")
df_dt = X_test.copy()
df_dt['Real'] = y_test
df_dt['Predicción Árbol'] = y_pred_dt.round(2)
st.dataframe(df_dt.head(10))

mse_dt = mean_squared_error(y_test, y_pred_dt)
rmse_dt = mse_dt ** 0.5
r2_dt = r2_score(y_test, y_pred_dt)
mae_dt = mean_absolute_error(y_test, y_pred_dt)

st.markdown("**📊 Métricas - Árbol de Decisión:**")
st.write(f"MSE: {mse_dt:.2f} | RMSE: {rmse_dt:.2f} | R²: {r2_dt:.2f} | MAE: {mae_dt:.2f}")

# =============================
st.subheader("🌲 Modelo: Random Forest")
model_rf = RandomForestRegressor(n_estimators=100, random_state=42)
model_rf.fit(X_train_scaled, y_train)
y_pred_rf = model_rf.predict(X_test_scaled)

mse_rf = mean_squared_error(y_test, y_pred_rf)
rmse_rf = mse_rf ** 0.5
r2_rf = r2_score(y_test, y_pred_rf)
mae_rf = mean_absolute_error(y_test, y_pred_rf)

st.markdown("**📊 Métricas - Random Forest:**")
st.write(f"MSE: {mse_rf:.2f} | RMSE: {rmse_rf:.2f} | R²: {r2_rf:.2f} | MAE: {mae_rf:.2f}")

fig, ax = plt.subplots(figsize=(8, 6))
sns.scatterplot(x=y_test, y=y_pred_rf, color='green', ax=ax)
ax.plot([y.min(), y.max()], [y.min(), y.max()], 'r--')
ax.set_title("Predicciones vs Realidad - Random Forest")
st.pyplot(fig)

# =============================
st.subheader("⚪ Modelo: SVR (Lineal)")
model_svr = SVR(kernel='linear')
model_svr.fit(X_train_scaled, y_train)
y_pred_svr = model_svr.predict(X_test_scaled)

mse_svr = mean_squared_error(y_test, y_pred_svr)
rmse_svr = mse_svr ** 0.5
r2_svr = r2_score(y_test, y_pred_svr)
mae_svr = mean_absolute_error(y_test, y_pred_svr)

st.markdown("**📊 Métricas - SVR Lineal:**")
st.write(f"MSE: {mse_svr:.2f} | RMSE: {rmse_svr:.2f} | R²: {r2_svr:.2f} | MAE: {mae_svr:.2f}")

# =============================
st.subheader("🟣 Modelo: SVR con Kernel RBF")
model_svr_rbf = SVR(kernel='rbf')
model_svr_rbf.fit(X_train_scaled, y_train)
y_pred_svr_rbf = model_svr_rbf.predict(X_test_scaled)

mse_rbf = mean_squared_error(y_test, y_pred_svr_rbf)
rmse_rbf = mse_rbf ** 0.5
r2_rbf = r2_score(y_test, y_pred_svr_rbf)
mae_rbf = mean_absolute_error(y_test, y_pred_svr_rbf)

st.markdown("**📊 Métricas - SVR RBF:**")
st.write(f"MSE: {mse_rbf:.2f} | RMSE: {rmse_rbf:.2f} | R²: {r2_rbf:.2f} | MAE: {mae_rbf:.2f}")

fig, ax = plt.subplots(figsize=(8, 6))
sns.scatterplot(x=y_test, y=y_pred_svr_rbf, color='purple', ax=ax)
ax.plot([y.min(), y.max()], [y.min(), y.max()], 'r--')
ax.set_title("Predicciones vs Realidad - SVR RBF")
st.pyplot(fig)


Writing 1_Regression.py


In [5]:
!mv 1_Regression.py pages/ #PARA QUE GUARDE LO NUEVO QUE HAGA

In [6]:
%%writefile 2_TAPNet.py

import streamlit as st
import pandas as pd
import numpy as np
import gdown
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from torch.utils.data import TensorDataset, DataLoader
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

# ========== CABECERA ==========
st.set_page_config(page_title="Clasificación Musical TAPNet", layout="wide")
st.title("📈 Clasificación de Popularidad Musical con TAPNet")

# ========== CARGA DE DATOS ==========
st.header("🔹 Carga de Datos")
file_id = '1jpW7VNRftEUjVj_yZCOBY6SgA5qa6J4T'
output = 'cleaned_music_population.csv'
url = f'https://drive.google.com/uc?id={file_id}'

if not os.path.exists(output):
    st.info("📥 Descargando archivo desde Google Drive...")
    gdown.download(url, output, quiet=False)

# Leer CSV
df = pd.read_csv(output)
st.success("✅ Archivo cargado correctamente.")
st.write("Vista previa del DataFrame:")
st.dataframe(df.head())

# ========== CLASIFICACION ==========
st.header("🔹 Clasificación de Popularidad")
df['pop_class'] = pd.qcut(df['popularity'], q=3, labels=[0, 1, 2]).astype(int)
st.write("Conteo de clases de popularidad:")
st.dataframe(df['pop_class'].value_counts().rename_axis("Clase").reset_index(name="Cantidad"))

# ========== PREPROCESAMIENTO ==========
X = df.drop(columns=['popularity', 'pop_class'])
y = df['pop_class']
X_encoded = pd.get_dummies(X)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_encoded)
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, stratify=y, random_state=42
)

# ========== MODELO ==========
class FeatureExtractor(nn.Module):
    def __init__(self, input_dim, hidden_dim=64, output_dim=32):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        return self.fc2(x)

class TAPNet(nn.Module):
    def __init__(self, input_dim, num_classes):
        super().__init__()
        self.feat_extractor = FeatureExtractor(input_dim)
        self.num_classes = num_classes

    def forward(self, x):
        return self.feat_extractor(x)

def get_prototypes(model, X_tensor, y_tensor, num_classes):
    model.eval()
    with torch.no_grad():
        features = model(X_tensor)
    return torch.stack([
        features[y_tensor == c].mean(dim=0) for c in range(num_classes)
    ])

def compute_logits(features, prototypes):
    return -torch.cdist(features, prototypes)

# Convertir a tensores
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.long)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.long)

# Crear loader y modelo
train_loader = DataLoader(TensorDataset(X_train_tensor, y_train_tensor), batch_size=32, shuffle=True)
input_dim = X_train.shape[1]
num_classes = len(np.unique(y))
model = TAPNet(input_dim=input_dim, num_classes=num_classes)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
loss_fn = nn.CrossEntropyLoss()

# ========== ENTRENAMIENTO ==========
st.header("🔹 Entrenamiento del Modelo TAPNet")
epoch_logs = []
model.train()
for epoch in range(30):
    epoch_loss = 0
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        features = model(X_batch)
        prototypes = get_prototypes(model, X_train_tensor, y_train_tensor, num_classes)
        logits = compute_logits(features, prototypes)
        loss = loss_fn(logits, y_batch)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    epoch_logs.append(f"Epoch {epoch+1}/30 - Pérdida: {epoch_loss:.4f}")

# Mostrar entrenamiento en desplegable
with st.expander("📂 Ver detalles del entrenamiento por época"):
    for log in epoch_logs:
        st.text(log)

# ========== EVALUACION ==========
model.eval()
with torch.no_grad():
    test_features = model(X_test_tensor)
    test_prototypes = get_prototypes(model, X_train_tensor, y_train_tensor, num_classes)
    test_logits = compute_logits(test_features, test_prototypes)
    y_pred = torch.argmax(test_logits, dim=1)

# ======== REPORTE DE CLASIFICACION ========
st.subheader("📋 Reporte de Clasificación")
report_dict = classification_report(y_test_tensor, y_pred, target_names=["Baja", "Media", "Alta"], output_dict=True)
report_df = pd.DataFrame(report_dict).transpose()
st.dataframe(report_df.style.format(precision=2))

# ======== MATRIZ DE CONFUSION ========
st.subheader("📊 Matriz de Confusión")
cm = confusion_matrix(y_test_tensor, y_pred)
fig, ax = plt.subplots()
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=["Baja", "Media", "Alta"],
            yticklabels=["Baja", "Media", "Alta"])
ax.set_xlabel("Predicción")
ax.set_ylabel("Real")
ax.set_title("Matriz de Confusión - TAPNet")
st.pyplot(fig)

# ========== PCA ==========
st.subheader("📌 Visualización PCA de Embeddings")
pca_result = PCA(n_components=2).fit_transform(test_features.numpy())
fig, ax = plt.subplots(figsize=(8, 6))
scatter = ax.scatter(pca_result[:, 0], pca_result[:, 1], c=y_test_tensor.numpy(), cmap='viridis', alpha=0.7)
legend = ax.legend(*scatter.legend_elements(), title="Clase")
ax.add_artist(legend)
ax.set_title("Visualización PCA de embeddings TAPNet")
ax.set_xlabel("Componente 1")
ax.set_ylabel("Componente 2")
ax.grid(True)
st.pyplot(fig)

# ========== t-SNE ==========
st.subheader("🎯 Visualización t-SNE de Embeddings")
tsne_result = TSNE(n_components=2, random_state=42, perplexity=30).fit_transform(test_features.numpy())
fig, ax = plt.subplots(figsize=(8, 6))
scatter = ax.scatter(tsne_result[:, 0], tsne_result[:, 1], c=y_test_tensor.numpy(), cmap='viridis', alpha=0.7)
legend = ax.legend(*scatter.legend_elements(), title="Clase")
ax.add_artist(legend)
ax.set_title("Visualización t-SNE de embeddings TAPNet")
ax.set_xlabel("Dimensión 1")
ax.set_ylabel("Dimensión 2")
ax.grid(True)
st.pyplot(fig)


Writing 2_TAPNet.py


In [7]:
!mv 2_TAPNet.py pages/ #PARA QUE GUARDE LO NUEVO QUE HAGA

# **Inicialización del Dashboard**

In [8]:
!wget https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared-linux-amd64
!mv cloudflared-linux-amd64 /usr/local/bin/cloudflared

#Ejecutar Streamlit
!streamlit run 0_👋_BIENVENIDOS.py &>/content/logs.txt & #Cambiar 0_👋_Hello.py por el nombre de tu archivo principal

#Exponer el puerto 8501 con Cloudflare Tunnel
!cloudflared tunnel --url http://localhost:8501 > /content/cloudflared.log 2>&1 &

#Leer la URL pública generada por Cloudflare
import time
time.sleep(5)  # Esperar que se genere la URL

import re
found_context = False  # Indicador para saber si estamos en la sección correcta

with open('/content/cloudflared.log') as f:
    for line in f:
        #Detecta el inicio del contexto que nos interesa
        if "Your quick Tunnel has been created" in line:
            found_context = True

        #Busca una URL si ya se encontró el contexto relevante
        if found_context:
            match = re.search(r'https?://\S+', line)
            if match:
                url = match.group(0)  #Extrae la URL encontrada
                print(f'Tu aplicación está disponible en: {url}')
                break  #Termina el bucle después de encontrar la URL



--2025-07-23 22:25:05--  https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
Resolving github.com (github.com)... 140.82.112.3
Connecting to github.com (github.com)|140.82.112.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://github.com/cloudflare/cloudflared/releases/download/2025.7.0/cloudflared-linux-amd64 [following]
--2025-07-23 22:25:05--  https://github.com/cloudflare/cloudflared/releases/download/2025.7.0/cloudflared-linux-amd64
Reusing existing connection to github.com:443.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/106867604/37d2bad8-a2ed-4b93-8139-cbb15162d81d?sp=r&sv=2018-11-09&sr=b&spr=https&se=2025-07-23T23%3A08%3A43Z&rscd=attachment%3B+filename%3Dcloudflared-linux-amd64&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2025-07-23T2